# 05 — Holarchy Visualization

This notebook demonstrates the data-structure side of holarchy
visualization: building node/edge payloads shaped for common
visualization libraries (graphology/sigma.js, NetworkX, yFiles).

The library does NOT ship rendering. It produces data structures;
downstream tools render them.


## Setup — a modest holarchy

One parent, three children, a couple of portals.


In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

ds.add_holon("urn:holon:program", "Program X")
ds.add_holon("urn:holon:domain-a", "Domain A", member_of="urn:holon:program")
ds.add_holon("urn:holon:domain-b", "Domain B", member_of="urn:holon:program")
ds.add_holon("urn:holon:service-1", "Service 1", member_of="urn:holon:domain-a")
ds.add_holon("urn:holon:service-2", "Service 2", member_of="urn:holon:domain-b")

# Inter-domain portal
ds.add_portal(
    "urn:portal:a-to-b",
    source_iri="urn:holon:domain-a",
    target_iri="urn:holon:domain-b",
    construct_query="CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
    label="A → B",
)

# Service portal
ds.add_portal(
    "urn:portal:s1-to-s2",
    source_iri="urn:holon:service-1",
    target_iri="urn:holon:service-2",
    construct_query="CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
    label="S1 → S2",
)


## Topology projection

`project_holarchy()` walks the registry and produces a
`ProjectedGraph` whose nodes are holons and whose edges are
`memberOf` + portal connections.


In [ ]:
topo = ds.project_holarchy()

print("Nodes (holons):")
for iri, node in topo.nodes.items():
    print(f"  {iri}  label={node.label!r}  kind={node.types}")

print()
print("Edges:")
for edge in topo.edges:
    print(f"  {edge.source}  --[{edge.predicate}]-->  {edge.target}")


## Console-shaped neighborhood (graphology-compatible)

For web consoles, `holon_neighborhood(iri, depth=N)` returns a BFS
subgraph shaped for sigma.js consumption via `to_graphology()`.
Covered in depth in `06_console_views`.


In [ ]:
neighborhood = ds.holon_neighborhood("urn:holon:domain-a", depth=2)
graphology = neighborhood.to_graphology()
print(f"nodes: {len(graphology['nodes'])}")
print(f"edges: {len(graphology['edges'])}")
print()
print("sample node:", graphology["nodes"][0])


## NetworkX export

`ProjectedGraph.to_dict()` produces a JSON-serializable mapping. For
NetworkX, walk the nodes/edges directly:


In [ ]:
# Requires networkx
# !pip install networkx

import networkx as nx

g = nx.DiGraph()
for iri, node in topo.nodes.items():
    g.add_node(iri, label=node.label, types=list(node.types))
for edge in topo.edges:
    if edge.predicate.endswith("memberOf"):
        # cga:memberOf points child → parent in RDF.
        # Reverse it so the digraph follows the containment
        # hierarchy top-down (parent → child), which is what
        # nx.descendants expects.
        g.add_edge(edge.target, edge.source, predicate="contains")
    else:
        # Portal edges (sourceHolon → targetHolon) keep their
        # original direction — they represent data flow.
        g.add_edge(edge.source, edge.target, predicate=edge.predicate)

print(f"NetworkX graph: {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")
print(f"program's descendants: {list(nx.descendants(g, 'urn:holon:program'))}")


## Where to go next

- `06_console_views` — shaped for operator consoles
- `08_scope_resolution` — find holons matching a predicate without
  fetching the full topology
